# OlmoEarth × FiftyOne — A Three-Part Earth-Observation Demo

This notebook demonstrates how to pair **[OlmoEarth](https://github.com/allenai/olmoearth_pretrain)** —
Allen AI's family of multimodal, spatio-temporal Earth-observation foundation models — with
**[FiftyOne](https://docs.voxel51.com/)** for interactive dataset curation and visualization.

We build **three** self-contained demos on a shared set of Sentinel-2 scenes:

| # | Demo | FiftyOne capability showcased |
|---|------|-------------------------------|
| 1 | **Unsupervised embedding exploration** | `compute_visualization` (UMAP), Embeddings panel, `compute_uniqueness` |
| 2 | **Labeled benchmark task** | classification labels, `compute_similarity`, sort-by-similarity, evaluation views |
| 3 | **Geospatial map view** | `GeoLocation` fields, the App's interactive Map panel |

**Model used as a frozen embedding extractor.** We never fine-tune here — OlmoEarth produces
embeddings, FiftyOne does the rest.

---

### Environment assumptions

- **Python 3.12** (OlmoEarth's recommended interpreter), in a **fresh virtual environment**.
- Runs on **CPU**, **Apple Silicon (MPS)**, or **CUDA** — OlmoEarth needs no specific GPU.
- A few GB of disk for the EuroSAT sample and model weights.

> **Recommended: run this in a dedicated virtual environment** so the dependency pins below don't
> disturb other projects. Create one and register it as a Jupyter kernel:
> ```bash
> python3.12 -m venv .venv
> source .venv/bin/activate            # Windows: .venv\Scripts\activate
> pip install --upgrade pip ipykernel
> python -m ipykernel install --user --name olmoearth-demo --display-name "olmoearth-demo"
> jupyter lab
> ```
> Then select the **olmoearth-demo** kernel. The first cell prints which interpreter you're on so
> you can confirm it's the venv before installing anything.

### Important: NumPy version constraint

This stack has a narrow NumPy window. Two key packages disagree:

| Package | NumPy requirement |
|---------|-------------------|
| `rasterio` 1.5.x | **≥ 2.0** |
| `numba` (via `umap-learn`) | **≤ 2.4** |

The version satisfying both is **`numpy==2.4.0`**, which the setup cell pins. In a fresh venv this
just works. If you're reusing an environment and `pip check` later flags some other package that
needs `numpy<2`, either remove it (if it's unused here) or use the alternative in the setup cell
(pin `rasterio<1.4` + `numpy==1.26.4`).

> **After the setup cell runs once, SKIP it for the rest of the session.** Re-running it can
> re-resolve NumPy and break the pin. Start re-runs from the "verify imports" cell below it.


## 0. Setup

We confirm which interpreter the kernel is using, install the OlmoEarth + geospatial packages,
then **pin NumPy to 2.4.0** so the whole stack agrees on one NumPy.


In [ ]:
import sys, os

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])

# Heuristic check that we're in a virtual environment (not the system Python).
in_venv = (
    sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    or "VIRTUAL_ENV" in os.environ
)
if in_venv:
    print(f"\n[ok] Running inside a virtual environment:\n     {sys.prefix}")
else:
    print(
        "\n[warning] This kernel does not appear to be running inside a virtual "
        "environment.\n          The dependency pins below will modify whatever "
        "environment this is.\n          Consider creating a dedicated venv (see the "
        "intro) and selecting its kernel."
    )

if sys.version_info[:2] != (3, 12):
    print(f"\n[note] Python {sys.version.split()[0]} detected; OlmoEarth recommends 3.12.")


In [ ]:
# --- One-time install + dependency resolution -----------------------------
# Run this cell ONCE, then RESTART THE KERNEL, then SKIP this cell for the rest
# of the session (re-running it can re-resolve NumPy and break the 2.4.0 pin).

# 1) FiftyOne + the Brain (embeddings, similarity, visualization).
%pip install -q --upgrade "fiftyone"

# 2) OlmoEarth + geospatial deps. olmoearth-pretrain pulls in torch;
#    rasterio/pyproj handle imagery + CRS math.
%pip install -q --upgrade "olmoearth-pretrain" "rasterio" "pyproj" "scikit-image"

# 3) Dimensionality reduction for FiftyOne's Embeddings panel.
#    pip package name is umap-learn; the import name is umap.
%pip install -q "umap-learn"

# 4) Pin the one NumPy that satisfies rasterio (>=2) AND numba (<=2.4): 2.4.0.
%pip install -q "numpy==2.4.0"

print("\n[done] Now: Kernel -> Restart Kernel, then run the verify cell below.")
print("       Do NOT re-run this cell after restarting.")

# --- If `pip check` later flags a numpy<2 conflict ------------------------
# In a fresh venv this won't happen. If you reused an environment and some
# other package pins numpy<2, either remove it (if unused here):
#   %pip uninstall -q -y <package>
# Or pin older rasterio instead, which works on numpy 1.26:
#   %pip install -q "rasterio<1.4" "numpy==1.26.4"


**Restart the kernel now** (Kernel → Restart Kernel), then run the verify cell. After a clean
restart, `import numpy` must report `2.4.0` *before* anything imports it.

In [ ]:
# Verify the whole stack agrees on one NumPy. This is the safe re-entry point
# for the session -- start here on any re-run instead of the install cell.
import numpy
print("numpy   ", numpy.__version__)          # expect 2.4.0
assert numpy.__version__.startswith("2.4"), (
    f"NumPy is {numpy.__version__}, expected 2.4.x. Re-run the install cell, "
    "then RESTART THE KERNEL before importing anything."
)

import rasterio;                 print("rasterio", rasterio.__version__)
import numba, umap;              print("numba/umap ok")
import torch;                    print("torch   ", torch.__version__)
from olmoearth_pretrain.model_loader import ModelID, load_model_from_id
print("olmoearth ok  (the 'olmo-core not installed' UserWarning is expected -- inference-only mode)")

import fiftyone, fiftyone.brain
print("fiftyone", fiftyone.__version__)
print("\n[ok] environment verified.")


In [ ]:
import os, sys, torch

def get_device():
    """Pick the best available device on macOS: MPS (Apple Silicon) -> CPU."""
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():           # for completeness if run elsewhere
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
print("Using device:", DEVICE)

# A scratch directory for downloaded imagery / cached arrays
WORK_DIR = os.path.abspath("./olmoearth_fiftyone_demo")
os.makedirs(WORK_DIR, exist_ok=True)
print("Working dir :", WORK_DIR)


## 1. Acquire a small Sentinel-2 dataset

For a fast, reproducible demo we use the **EuroSAT** RGB+multispectral land-cover dataset, which
OlmoEarth itself is benchmarked on (`m-eurosat` in the paper). EuroSAT is:

- **Sentinel-2 L2A** imagery — exactly OlmoEarth's primary modality,
- **labeled** into 10 land-cover classes (great for Demos 2 & 3),
- **georeferenced**, so each tile carries a real-world coordinate (Demo 3),
- **small** (64×64 px tiles) — ideal for OlmoEarth's recommended 1×1–128×128 input range.

We pull a modest sample so the whole notebook runs in minutes on a laptop. If you have a
`.SAFE` Sentinel-2 scene of your own (see the OlmoEarth Inference Quickstart), Section 5 shows how
to swap it in.

> **Already downloaded the archive?** If `EuroSATallBands.zip` is sitting in your `~/Downloads`
> folder, the cell below detects it automatically, **moves** it into the working directory, and
> unzips it — no re-download. Edit `PREDOWNLOADED_CANDIDATES` if your copy lives elsewhere.

> **Bands.** OlmoEarth's `SENTINEL2_L2A` modality expects **12 bands in a specific order**. EuroSAT
> ships all 13 Sentinel-2 bands; we select and order them to match `Modality.SENTINEL2_L2A.band_order`.


In [ ]:
import numpy as np

# How many tiles to process. Keep small for a laptop demo; raise for richer plots.
N_SAMPLES = 300
RANDOM_SEED = 51
rng = np.random.default_rng(RANDOM_SEED)

# EuroSAT multispectral (all 13 Sentinel-2 bands) is distributed as .tif tiles.
# We download a single archive and sample from it.
EUROSAT_MS_URL = "https://madm.dfki.de/files/sentinel/EuroSATallBands.zip"
print("EuroSAT (all bands) source:", EUROSAT_MS_URL)
print(
    "\nNote: this archive is ~2 GB. If your network blocks it, see Section 5 for the\n"
    "single-scene fallback used in the OlmoEarth quickstart (a Seattle .SAFE scene)."
)


In [ ]:
import urllib.request, zipfile, pathlib, ssl, shutil

archive = pathlib.Path(WORK_DIR) / "EuroSATallBands.zip"
extract_root = pathlib.Path(WORK_DIR) / "eurosat_ms"

# If you already downloaded the archive (e.g. via your browser or curl), it most
# likely landed in ~/Downloads. Check there first and move it into WORK_DIR so we
# never re-download. Add more candidate locations here if needed.
PREDOWNLOADED_CANDIDATES = [
    pathlib.Path.home() / "Downloads" / "EuroSATallBands.zip",
    pathlib.Path.cwd() / "EuroSATallBands.zip",
]

def adopt_predownloaded(dest):
    """If a valid zip exists in a known location, move it to dest. Returns True if adopted."""
    if dest.exists() and dest.stat().st_size > 0:
        return True
    for cand in PREDOWNLOADED_CANDIDATES:
        if cand.exists() and cand.stat().st_size > 0 and cand.resolve() != dest.resolve():
            print(f"[found] pre-downloaded archive at {cand} ({cand.stat().st_size/1e9:.2f} GB)")
            print(f"[move]  -> {dest}")
            shutil.move(str(cand), str(dest))   # move (not copy) to avoid duplicating ~2 GB
            return True
    return False

# macOS + Python often lacks a wired-up CA bundle, causing
# CERTIFICATE_VERIFY_FAILED. Use certifi's bundle explicitly so urllib trusts
# the server. (Setting SSL_CERT_FILE alone is unreliable mid-process.)
def _ssl_context():
    try:
        import certifi
        return ssl.create_default_context(cafile=certifi.where())
    except Exception:
        return ssl.create_default_context()

def download(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[cached] {dest.name} ({dest.stat().st_size/1e9:.2f} GB)")
        return
    print(f"[download] {url}\n        -> {dest}  (~2 GB, be patient)")
    ctx = _ssl_context()
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    # Stream to disk so a 2 GB file doesn't sit in memory.
    with urllib.request.urlopen(req, context=ctx) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f, length=1024 * 1024)

try:
    # Prefer an already-downloaded copy (e.g. from ~/Downloads); else fetch it.
    if not adopt_predownloaded(archive):
        download(EUROSAT_MS_URL, archive)

    if not extract_root.exists():
        print("[unzip] extracting (one-time)...")
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(extract_root)
    print("[ok] EuroSAT ready at", extract_root)
    HAVE_EUROSAT = True
except Exception as e:
    print("[warn] EuroSAT download/extract failed:", repr(e))
    print("       Two robust fallbacks:")
    print("       (a) Download outside Python (uses the system trust store):")
    print(f"           cd {WORK_DIR}")
    print("           curl -L -o EuroSATallBands.zip \\")
    print("                https://madm.dfki.de/files/sentinel/EuroSATallBands.zip")
    print("           # then re-run this cell -- it will use the cached file.")
    print("       (b) Use the Section 5 single-scene path (no labels).")
    HAVE_EUROSAT = False


In [ ]:
# Index the extracted .tif tiles: directory name == class label.
import glob

tif_paths, labels = [], []
if HAVE_EUROSAT:
    # The archive extracts to .../ds/images/remote_sensing/otherDatasets/sentinel_2/tif/<CLASS>/*.tif
    # but layouts vary by mirror; search broadly for class subfolders of .tif files.
    candidates = glob.glob(str(extract_root / "**" / "*.tif"), recursive=True)
    for p in candidates:
        cls = pathlib.Path(p).parent.name
        tif_paths.append(p)
        labels.append(cls)
    classes = sorted(set(labels))
    print(f"[ok] found {len(tif_paths)} tiles across {len(classes)} classes:")
    print("    ", classes)

    # Subsample N_SAMPLES, stratified-ish by shuffling then trimming
    idx = rng.permutation(len(tif_paths))[:N_SAMPLES]
    tif_paths = [tif_paths[i] for i in idx]
    labels    = [labels[i]    for i in idx]
    print(f"[ok] sampled {len(tif_paths)} tiles for the demo")
else:
    classes = []
    print("[skip] EuroSAT not available; Section 5 fallback will supply data.")


## 2. Load OlmoEarth and define an embedding function

We load **OlmoEarth v1 Base** as a frozen encoder. The encoder returns per-token features shaped
`(B, H', W', T, S, D)`; following the official quickstart we **mean-pool over the timestep and
band-set dimensions** to get a `(B, H', W', D)` feature map, then **mean-pool spatially** to obtain
one embedding vector per tile — the natural input to FiftyOne's brain methods.

Key correctness details baked into the helper:

- **Band selection/ordering** to match `Modality.SENTINEL2_L2A.band_order`.
- **Normalization** via OlmoEarth's `Normalizer(Strategy.COMPUTED)`.
- **Layout** `B,H,W,T,C` and a mask tensor of shape `B,H,W,T,S` (S = 3 band-sets for S2).
- Inputs kept within the model's recommended **≤128×128** spatial size.


In [ ]:
from olmoearth_pretrain.model_loader import ModelID, load_model_from_id
from olmoearth_pretrain.datatypes import MaskedOlmoEarthSample, MaskValue
from olmoearth_pretrain.data.constants import Modality
from olmoearth_pretrain.data.normalize import Normalizer, Strategy

# --- choose model size ----------------------------------------------------
# Nano/Tiny are fastest on CPU/MPS; Base matches the paper's headline numbers.
MODEL_ID = ModelID.OLMOEARTH_V1_BASE     # try OLMOEARTH_V1_NANO for a quick first pass
PATCH_SIZE = 8                            # 1..8; smaller = better but slower

print(f"[load] {MODEL_ID} ...")
model = load_model_from_id(MODEL_ID)
model.eval().to(DEVICE)
normalizer = Normalizer(Strategy.COMPUTED)

S2_BAND_ORDER = Modality.SENTINEL2_L2A.band_order
print("[ok] model loaded")
print("S2 band order expected by OlmoEarth:", S2_BAND_ORDER, f"({len(S2_BAND_ORDER)} bands)")


In [ ]:
import rasterio
from rasterio.enums import Resampling

# EuroSAT 13-band .tif band indexing (1-based, GDAL): B01,B02,...,B08,B8A,B09,B10,B11,B12
# Map OlmoEarth's expected band names to EuroSAT .tif band positions.
EUROSAT_BAND_TO_INDEX = {
    "B01": 1, "B02": 2, "B03": 3, "B04": 4, "B05": 5, "B06": 6, "B07": 7,
    "B08": 8, "B8A": 9, "B09": 10, "B10": 11, "B11": 12, "B12": 13,
}

def read_eurosat_tile(path, size=64):
    """Read a EuroSAT .tif into (H, W, C) with bands ordered for OlmoEarth.

    Returns (image_hwc float32, rgb_uint8 for visualization, (lon, lat) or None).
    """
    with rasterio.open(path) as src:
        # Read every band we need, resampled to `size` if necessary.
        bands = []
        for bname in S2_BAND_ORDER:
            gidx = EUROSAT_BAND_TO_INDEX.get(bname)
            if gidx is None or gidx > src.count:
                # Band not present (shouldn't happen for 13-band EuroSAT); fill zeros.
                bands.append(np.zeros((size, size), dtype=np.float32))
                continue
            arr = src.read(
                gidx, out_shape=(size, size), resampling=Resampling.bilinear
            ).astype(np.float32)
            bands.append(arr)
        image = np.stack(bands, axis=-1)  # (H, W, C)

        # Best-effort geolocation: center of the tile reprojected to EPSG:4326.
        lonlat = None
        try:
            from rasterio.warp import transform as warp_transform
            cx, cy = src.xy(src.height // 2, src.width // 2)
            xs, ys = warp_transform(src.crs, "EPSG:4326", [cx], [cy])
            lonlat = (float(xs[0]), float(ys[0]))
        except Exception:
            lonlat = None

    # RGB preview (B04,B03,B02) -> stretch to 0..255 for FiftyOne media
    def _band(b):
        return image[..., S2_BAND_ORDER.index(b)] if b in S2_BAND_ORDER else image[..., 0]
    rgb = np.stack([_band("B04"), _band("B03"), _band("B02")], axis=-1)
    lo, hi = np.percentile(rgb, 2), np.percentile(rgb, 98)
    rgb = np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)
    rgb_uint8 = (rgb * 255).astype(np.uint8)

    return image, rgb_uint8, lonlat


In [ ]:
@torch.no_grad()
def olmoearth_embed(image_hwc, timestamp=(15, 6, 2024)):
    """Embed a single (H, W, C) Sentinel-2 tile into a 1-D OlmoEarth feature vector.

    timestamp: (day_of_month 1-31, month 0-11, year)
    """
    H, W, C = image_hwc.shape
    # OlmoEarth normalize() expects B,H,W,T,C
    arr = image_hwc[None, :, :, None, :].astype(np.float32)   # (1,H,W,1,C)
    arr = normalizer.normalize(Modality.SENTINEL2_L2A, arr)

    x = torch.tensor(arr, dtype=torch.float32, device=DEVICE)
    # Mask shape is B,H,W,T,S with S=3 band-sets for Sentinel-2; ONLINE_ENCODER = "feed to encoder".
    mask = torch.ones((1, H, W, 1, 3), dtype=torch.float32, device=DEVICE) * MaskValue.ONLINE_ENCODER.value
    ts = torch.tensor(timestamp, device=DEVICE)[None, None, :]   # (1,1,3)

    sample = MaskedOlmoEarthSample(
        sentinel2_l2a=x,
        sentinel2_l2a_mask=mask,
        timestamps=ts,
    )
    out = model.encoder(sample, fast_pass=True, patch_size=PATCH_SIZE)["tokens_and_masks"]
    feats = out.sentinel2_l2a              # (B, H', W', T, S, D)
    pooled = feats.mean(dim=[3, 4])        # -> (B, H', W', D)
    vec = pooled.mean(dim=[1, 2])          # spatial mean -> (B, D)
    return vec.squeeze(0).float().cpu().numpy()

# quick smoke test on random data shaped like a tile
_test = np.random.randn(64, 64, len(S2_BAND_ORDER)).astype(np.float32)
_emb = olmoearth_embed(_test)
print("[ok] embedding dim:", _emb.shape)


## 3. Build the FiftyOne dataset

We render each tile's RGB preview to a PNG (so the App has something to display), attach the
ground-truth land-cover **label**, a **GeoLocation** point, and compute the **OlmoEarth embedding**.
Everything lands on one `fo.Dataset` that all three demos share.


In [ ]:
import fiftyone as fo
from PIL import Image
from tqdm.auto import tqdm

DATASET_NAME = "olmoearth-eurosat-demo"
if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)          # clean re-runs
dataset = fo.Dataset(DATASET_NAME, persistent=True)

preview_dir = pathlib.Path(WORK_DIR) / "previews"
preview_dir.mkdir(exist_ok=True)

samples, embeddings = [], []

iterator = list(zip(tif_paths, labels)) if HAVE_EUROSAT else []
for i, (path, cls) in enumerate(tqdm(iterator, desc="Embedding tiles")):
    try:
        image_hwc, rgb_uint8, lonlat = read_eurosat_tile(path, size=64)
    except Exception as e:
        print("skip", path, repr(e)); continue

    # Save an RGB preview PNG for the App to display.
    png_path = preview_dir / f"tile_{i:05d}.png"
    Image.fromarray(rgb_uint8).save(png_path)

    emb = olmoearth_embed(image_hwc)
    embeddings.append(emb)

    s = fo.Sample(filepath=str(png_path))
    s["ground_truth"] = fo.Classification(label=cls)
    s["olmoearth_embedding"] = emb.tolist()      # also store on-sample for reuse
    if lonlat is not None:
        s["location"] = fo.GeoLocation(point=[lonlat[0], lonlat[1]])  # [lon, lat]
    s["source_tif"] = path
    samples.append(s)

if samples:
    dataset.add_samples(samples)
    embeddings = np.stack(embeddings)
    print(f"[ok] dataset built: {len(dataset)} samples, embedding matrix {embeddings.shape}")
else:
    print("[skip] no samples added (EuroSAT unavailable). Run Section 5 instead.")


In [ ]:
# Persist the embedding matrix aligned to dataset sample order for the brain methods.
if len(dataset):
    # Re-pull embeddings in the dataset's canonical order to stay aligned.
    embeddings = np.array([s["olmoearth_embedding"] for s in dataset.iter_samples()])
    print("Embedding matrix (aligned):", embeddings.shape)


## Demo 1 — Unsupervised embedding exploration

We reduce the OlmoEarth embeddings to 2-D with **UMAP** and open the **Embeddings panel**. Because
OlmoEarth is a strong EO foundation model, tiles should cluster by land cover *without ever showing
the model a label*. We also run `compute_uniqueness` to surface rare / anomalous tiles.


In [ ]:
import fiftyone.brain as fob

if len(dataset):
    # Custom embeddings -> pass the precomputed matrix directly.
    viz = fob.compute_visualization(
        dataset,
        embeddings=embeddings,          # our OlmoEarth vectors
        method="umap",
        num_dims=2,
        brain_key="olmoearth_umap",
        verbose=True,
    )
    print("[ok] UMAP visualization stored under brain_key='olmoearth_umap'")


In [ ]:
if len(dataset):
    # Uniqueness ranks tiles by how distinct they are in embedding space.
    fob.compute_uniqueness(
        dataset,
        embeddings=embeddings,
        uniqueness_field="olmoearth_uniqueness",
    )
    print("[ok] uniqueness computed -> field 'olmoearth_uniqueness'")
    most_unique = dataset.sort_by("olmoearth_uniqueness", reverse=True).limit(5)
    print("Top-5 most unique tiles (label, uniqueness):")
    for s in most_unique:
        print(f"  {s.ground_truth.label:<22} {s.olmoearth_uniqueness:.4f}")


In [ ]:
# Launch the App. In the App: add an "Embeddings" panel and color by ground_truth
# to see unsupervised clusters; add a "Map" panel for Demo 3.
if len(dataset):
    session = fo.launch_app(dataset)
    print("App launched. Open the Embeddings panel and color by 'ground_truth'.")


## Demo 2 — Labeled benchmark task + similarity search

EuroSAT is the `m-eurosat` benchmark from the OlmoEarth paper. Here we:

1. Build a **similarity index** on the OlmoEarth embeddings with `compute_similarity`.
2. **Sort by similarity** to a query tile — visually confirming the embeddings are semantically
   meaningful (a "Forest" query should retrieve forests).
3. Optionally train a tiny **k-NN probe** on the embeddings to reproduce the paper's *linear/kNN
   probing* protocol and report accuracy — a quantitative measure of embedding quality.


In [ ]:
if len(dataset):
    sim = fob.compute_similarity(
        dataset,
        embeddings=embeddings,
        brain_key="olmoearth_sim",
    )
    print("[ok] similarity index stored under brain_key='olmoearth_sim'")

    # Pick a query tile and retrieve its nearest neighbors.
    query_id = dataset.first().id
    query_label = dataset.first().ground_truth.label
    knn_view = dataset.sort_by_similarity(query_id, k=10, brain_key="olmoearth_sim")
    print(f"\nQuery tile label: {query_label}")
    print("Nearest-neighbor labels:")
    for s in knn_view:
        print("   ", s.ground_truth.label)
    # In the App, set the view to inspect retrieved tiles visually:
    if 'session' in dir():
        session.view = knn_view


In [ ]:
# Quantitative check: kNN probe on OlmoEarth embeddings (mirrors the paper's kNN protocol).
if len(dataset):
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report

    y = np.array([s.ground_truth.label for s in dataset.iter_samples()])
    X = embeddings

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
    )
    knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
    knn.fit(X_tr, y_tr)
    y_pred = knn.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    print(f"[result] kNN (k=5, cosine) accuracy on held-out EuroSAT tiles: {acc:.3f}")
    print("\n", classification_report(y_te, y_pred))


In [ ]:
# Write predictions back so we can use FiftyOne's evaluation views.
if len(dataset):
    # Predict for ALL samples (demo-grade; reuses the same knn fit above).
    preds = knn.predict(X)
    confs = knn.predict_proba(X).max(axis=1)
    for s, p, c in zip(dataset.iter_samples(autosave=True), preds, confs):
        s["knn_pred"] = fo.Classification(label=str(p), confidence=float(c))

    results = dataset.evaluate_classifications(
        "knn_pred", gt_field="ground_truth", eval_key="knn_eval", method="simple"
    )
    results.print_report()
    # In the App you can now filter to mistakes: knn_pred.label != ground_truth.label


## Demo 3 — Geospatial map view

Each EuroSAT tile carries a real-world coordinate, stored above as a `fo.GeoLocation`. FiftyOne's
**Map panel** plots those points on an interactive map; selecting a region on the map filters the
samples (and vice-versa), and you can color points by `ground_truth` or by `olmoearth_uniqueness`.

> **The Map panel needs a free Mapbox token** (public token, starts with `pk.`). Get one at
> [account.mapbox.com](https://account.mapbox.com/access-tokens/). The token must be set under the
> **map plugin** key — `plugins.map.mapboxAccessToken` — *not* the top-level
> `mapbox_access_token`, which the Map panel does not read. The cell below sets it correctly. For a
> permanent setting, add it to `~/.fiftyone/app_config.json`:
> ```json
> {"plugins": {"map": {"mapboxAccessToken": "pk.your_token_here"}}}
> ```
> then restart the kernel.


In [ ]:
# Set your Mapbox public token here, or via the MAPBOX_TOKEN environment variable.
MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN", "")   # or paste "pk.your_token_here"

if len(dataset):
    n_geo = len(dataset.exists("location"))
    print(f"{n_geo}/{len(dataset)} samples have a GeoLocation point.")

    if MAPBOX_TOKEN:
        # The Map plugin reads plugins.map.mapboxAccessToken (NOT mapbox_access_token).
        fo.app_config.plugins = {"map": {"mapboxAccessToken": MAPBOX_TOKEN}}
        print("[ok] Mapbox token configured for the Map panel.")
    else:
        print("[note] No Mapbox token set. Add one above (or export MAPBOX_TOKEN) and re-launch.")

    # Launch fresh so the App server picks up the token (a browser refresh is not enough).
    session = fo.launch_app(dataset)
    print("Add a 'Map' panel in the App and color points by 'ground_truth'.")


In [ ]:
# Region filter example: restrict to a geographic bounding box and save it as a view.
# Demonstrates the map <-> grid <-> embeddings link: load this view and all three update.
# Box here is a rough Sweden bbox [lon, lat]; change the corners for any region.
if len(dataset):
    region_poly = [[
        [11.0, 55.0],   # SW corner [lon, lat]
        [24.0, 55.0],   # SE
        [24.0, 69.0],   # NE
        [11.0, 69.0],   # NW
        [11.0, 55.0],   # close the ring
    ]]
    # geo_within resolves the GeoLocation field internally (handles the point layout for you).
    region_view = dataset.geo_within(region_poly, location_field="location")
    n = region_view.count()
    print(f"samples in region: {n}")

    if n:
        dataset.save_view("region", region_view, overwrite=True)
        session.view = dataset.load_saved_view("region")
        print("Saved + loaded view 'region'. Open the Map and Embeddings panels to see them sync.")
    else:
        print("[note] No tiles fell in this bounding box for the current sample. "
              "Widen the box (e.g. central Europe: lon 2-15, lat 45-55) or raise N_SAMPLES.")


## 4. Bonus — compare model sizes (v1 vs v1.2 story)

The OlmoEarth v1.2 report's headline is that the smaller-compute models match v1. You can make that
tangible in FiftyOne by embedding the **same tiles** with a second model size and adding a *second*
UMAP visualization under a different `brain_key`; the Embeddings panel lets you flip between them.


In [ ]:
COMPARE = False   # set True to run the second pass (slower)

if COMPARE and len(dataset):
    alt_id = ModelID.OLMOEARTH_V1_NANO
    print(f"[load] {alt_id}")
    alt_model = load_model_from_id(alt_id)
    alt_model.eval().to(DEVICE)

    @torch.no_grad()
    def embed_with(m, image_hwc, timestamp=(15, 6, 2024)):
        H, W, C = image_hwc.shape
        arr = normalizer.normalize(
            Modality.SENTINEL2_L2A, image_hwc[None, :, :, None, :].astype(np.float32)
        )
        x = torch.tensor(arr, dtype=torch.float32, device=DEVICE)
        mask = torch.ones((1, H, W, 1, 3), device=DEVICE) * MaskValue.ONLINE_ENCODER.value
        ts = torch.tensor(timestamp, device=DEVICE)[None, None, :]
        sample = MaskedOlmoEarthSample(sentinel2_l2a=x, sentinel2_l2a_mask=mask, timestamps=ts)
        out = m.encoder(sample, fast_pass=True, patch_size=PATCH_SIZE)["tokens_and_masks"]
        return out.sentinel2_l2a.mean(dim=[3, 4]).mean(dim=[1, 2]).squeeze(0).float().cpu().numpy()

    alt_embs = []
    for s in tqdm(dataset.iter_samples(), total=len(dataset), desc="Nano embeddings"):
        img, _, _ = read_eurosat_tile(s["source_tif"], size=64)
        alt_embs.append(embed_with(alt_model, img))
    alt_embs = np.stack(alt_embs)

    fob.compute_visualization(
        dataset, embeddings=alt_embs, method="umap",
        num_dims=2, brain_key="olmoearth_nano_umap",
    )
    print("[ok] second visualization stored under 'olmoearth_nano_umap' — compare in the App.")


## 5. Fallback: single `.SAFE` Sentinel-2 scene (no EuroSAT)

If the EuroSAT archive is unavailable, you can still demo the **embedding pipeline** on the
Seattle scene from the OlmoEarth quickstart. This won't give you labels (Demo 2) but exercises the
model end-to-end and produces a tiled embedding map you can explore.

```bash
wget https://storage.googleapis.com/ai2-rslearn-projects-data/artifacts/example_sentinel2_l2a_scene_of_seattle.zip
unzip example_sentinel2_l2a_scene_of_seattle.zip
```

Then read the `.SAFE` bands at 10 m, tile the scene into 64×64 windows, embed each window with
`olmoearth_embed`, and add them as FiftyOne samples (carrying each window's geographic centroid as a
`GeoLocation`). The per-tile embeddings flow into Demos 1 and 3 exactly as above. The reading code
mirrors the official quickstart: read every band in `Modality.SENTINEL2_L2A.band_order` via a
`WarpedVRT` keyed to the B02 transform, then `transpose` to `B,H,W,T,C`.


In [ ]:
# Sketch only — uncomment and adapt paths if running the fallback.
# import glob, rasterio
# from rasterio.vrt import WarpedVRT
# from rasterio.enums import Resampling
#
# fnames = [glob.glob(f"*.SAFE/GRANULE/*/IMG_DATA/*/*_{b}_*.jp2")[0]
#           for b in Modality.SENTINEL2_L2A.band_order]
# with rasterio.open(fnames[0]) as src:
#     crs, transform = src.crs, src.transform
# W = H = 512
# scene = np.zeros((len(fnames), H, W), dtype=np.int32)
# for bi, fn in enumerate(fnames):
#     with rasterio.open(fn) as src, WarpedVRT(src, crs=crs, transform=transform,
#                                              width=W, height=H,
#                                              resampling=Resampling.bilinear) as vrt:
#         scene[bi] = vrt.read(1)
# scene = scene.transpose(1, 2, 0)        # H,W,C
# # Tile into 64x64 windows, embed each with olmoearth_embed(window), add to a fo.Dataset.
print("Fallback sketch — see comments above.")


## 6. Cleanup & notes

```python
# fo.delete_dataset("olmoearth-eurosat-demo")   # if you don't want it to persist
```

**Practical reminders**

- **License.** OlmoEarth ships under the *OlmoEarth Artifact License*, which restricts military,
  defense-related, and extractive-industry use. This demo is research/education — keep within those terms.
- **Bands & order matter.** Feeding bands out of `Modality.SENTINEL2_L2A.band_order` silently degrades
  embeddings. The helper enforces the order; double-check it for any new data source.
- **Patch size.** Smaller `PATCH_SIZE` (down to 1) improves quality at the cost of compute; 8 is a fast
  default for a laptop.
- **Device.** On Apple Silicon, MPS gives a solid speedup over CPU. Some torch ops occasionally fall
  back to CPU on MPS — that's expected and harmless here.
- **Scaling up.** For thousands of tiles, pass `create_index=True` to `compute_visualization` /
  `compute_similarity` for efficient lassoing and queries in the App.
